# 04 — Calibration gán nhãn và đo độ đồng thuận

**Vì sao cần calibration**

Trước khi gán nhãn hàng nghìn review, cần biết guideline 4 nhãn (BUG / FEATURE / UX / OTHER) đã đủ rõ để hai người đọc cùng một review
thì ra cùng một nhãn hay chưa. Nếu chưa, mọi chỉ số F1 về sau đều thiếu cơ sở, vì nhãn tham chiếu không đáng tin. Vì vậy một vòng
calibration được thực hiện với hai người gán độc lập, rồi đo độ đồng thuận (Q10).

**Mẫu và hai batch**

Lấy ngẫu nhiên 150 review Shopee (seed 42) từ corpus trung gian và chia thành hai batch, mỗi batch 75 review, đúng như đề cương §7.2.1.
Batch 1 dùng để hai người gán, xem lại những chỗ lệch nhau và sửa guideline; batch 2 gán bằng guideline đã sửa và là batch quyết định
guideline đã đạt hay chưa.

**Bảng gán**

Bảng gán chỉ có `review_id` và nội dung review (bản gốc đã ẩn danh hóa, giữ nguyên emoji và viết tắt). Số sao, ngày và phiên bản app
được cố ý ẩn để người gán không bị số sao dẫn dắt (Q17).

**Cột `ambiguous` và `note`**

Ngoài cột nhãn còn có cột `ambiguous` (đánh 1 khi review có hai ý ngang nhau, khó chọn ý chính) và cột `note` để ghi lý do khi phân vân
— phần ghi chú này rất có ích khi họp sửa guideline.

**Gán độc lập**

Hai người làm trên hai bản sao Google Sheets riêng và không xem bản của nhau.

**Cổng κ**

Ngưỡng là Cohen's κ ≥ 0.6 trên batch 2. κ chỉ là điều kiện cần: các ca lệch giữa FEATURE và UX, và giữa nhóm actionable với OTHER
(tương ứng tầng 1 của mô hình), được xem cụ thể, vì đây là hai chỗ dễ nhầm nhất.

**Kết quả batch 1**

Sau batch 1 (κ = 0,671, 9 ca lệch), hai người đã họp và guideline được cập nhật lên v1.1 với 5 quy tắc phân xử R1–R5 và quy tắc đánh
`ambiguous` (xem `00_protocol/edge_cases_log.md`); batch 2 được gán bằng guideline v1.1.

**Dùng lại calibration**

150 review calibration được ghi vào manifest để loại khỏi Hold-out sau này; sau khi thống nhất nhãn, chúng được đưa vào Train (Q9).
Phần xử lý nằm trong `src/annotation/calibration.py` và `src/annotation/agreement.py`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
from src.utils.kaggle_paths import input_dir
from src.utils.seeds import set_seed
from src.preprocessing.clean import load_jsonl
from src.annotation.calibration import (draw_calibration, export_blank, write_manifest, INSTRUCTIONS,
                                        FEATURE_UX_KEYWORDS, draw_keyword_set, write_keyword_manifest)
from src.annotation.agreement import (merge, agreement, disagreements_by_pair, gate_checklist,
                                      build_finalize_rows, write_finalize)
set_seed(42)
TAG = 'T0_2026-10-06'
SOURCE = os.path.join(input_dir('interim'), f'shopee_clean_{TAG}.jsonl')
CAL_DIR = os.path.abspath(os.path.join('..', '07_human_eval', 'calibration'))
MANIFEST = os.path.join(input_dir('manifests'), f'calibration_manifest_{TAG}.json')

## Rút mẫu và xuất bảng gán nhãn

In [2]:
records = load_jsonl(SOURCE)
batches = draw_calibration(records, n=150, n_batches=2, seed=42)
sheets = {b: os.path.join(CAL_DIR, f'calibration_{b}_blank.csv') for b in batches}
if all(os.path.exists(p) for p in sheets.values()) and os.path.exists(MANIFEST):
    print('Bảng gán nhãn và manifest đã tồn tại — giữ nguyên (không ghi đè).')
else:
    for b, rows in batches.items():
        export_blank(rows, sheets[b])
    write_manifest(batches, SOURCE, sheets, MANIFEST, seed=42)
with open(MANIFEST, encoding='utf-8') as f:
    manifest = json.load(f)
for b, info in manifest['batches'].items():
    print(f"{b}: {len(info['review_ids'])} review → {info['sheet']}")
print('Manifest:', MANIFEST)
print()
print(INSTRUCTIONS)

Bảng gán nhãn và manifest đã tồn tại — giữ nguyên (không ghi đè).
batch1: 75 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\calibration\calibration_batch1_blank.csv
batch2: 75 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\calibration\calibration_batch2_blank.csv
Manifest: C:\Workspace\kltn-vn-app-review-requirements\01_data\manifests\calibration_manifest_T0_2026-10-06.json

Hướng dẫn cho người gán nhãn
- Nhãn hợp lệ ở cột `label`: BUG / FEATURE / UX / OTHER (theo `00_protocol/annotation_guideline_v1.md` và `decision_tree.md`).
- Cột `ambiguous`: điền 1 nếu review có hai ý actionable độc lập ngang nhau, không xác định được ý chính; để trống nếu không.
  Vẫn chọn MỘT nhãn tốt nhất ở cột `label`.
- Cột `note`: ghi ngắn lý do khi phân vân.
- Mỗi người làm trên BẢN SAO Google Sheets riêng (A và B), gán độc lập, không xem bản của người kia, không thảo luận khi đang gán.
- Cột `label` dùng Data validation dạng dropdown với 4 giá trị trên; cột `ambigu

In [3]:
# Xem nhanh vài dòng của batch 1 (đúng như người gán sẽ thấy)
import pandas as pd
pd.read_csv(sheets['batch1'], encoding='utf-8-sig').head(8)

,review_id,text,label,ambiguous,note
0,75db5c2c-46f8-4339-afaf-2f5ae33ba3f5,tốt,NaN,NaN,NaN
1,93d24e85-3896-4e63-97db-d8be6af3967f,trả lại cái lựa chọn đơn vị vận chuyển đi tại ...,NaN,NaN,NaN
2,3966135b-6b5c-4722-a125-469f65a18d40,tìm kiếm trên shoppee bị cái gì mà ko ra v cứ ...,NaN,NaN,NaN
3,45484b4a-c5eb-4bda-8547-7d2185b247de,ngon,NaN,NaN,NaN
4,b6d4b63e-9169-4f1c-935f-1ad4cb2cca10,cho voucher rồi lúc mua thì bảo là voucher ảo ...,NaN,NaN,NaN
5,4958811e-744a-4a0c-aa06-edc68b6fc1bc,chính sách trả hàng cực kỳ tệ. người bán gửi s...,NaN,NaN,NaN
6,fb0c4bbc-4705-402a-b569-25afa2dc25c7,"mỗi lần vào là vào thẳng Live & video , tôi kh...",NaN,NaN,NaN
7,67b0d008-530c-4681-8001-a253c98f4548,nó cứ bị dính m02 đồ,NaN,NaN,NaN


## Đo độ đồng thuận

**Điều kiện chạy**

Phần này chỉ có kết quả sau khi hai người gán xong và các file `calibration_<batch>_A.csv`, `calibration_<batch>_B.csv` đã được tải về
`07_human_eval/calibration/`; trước đó notebook chỉ báo đang chờ.

**Vai trò của hai batch**

Kết quả batch 1 dùng để thảo luận, còn batch 2 dùng để quyết định.

In [4]:
GATE_BATCH = 'batch2'
results = {}
for b in manifest['batches']:
    pa, pb = (os.path.join(CAL_DIR, f'calibration_{b}_{w}.csv') for w in ('A', 'B'))
    if not (os.path.exists(pa) and os.path.exists(pb)):
        print(f'{b}: đang chờ {os.path.basename(pa)} và {os.path.basename(pb)}')
        continue
    m = merge(pa, pb, expected_ids=manifest['batches'][b]['review_ids'])
    if m['problems']:
        print(f'{b}: cần sửa tệp gán nhãn trước khi đo:')
        for pr in m['problems']:
            print('   -', pr)
        continue
    results[b] = {'df': m['df'], 'res': agreement(m['df'])}
    results[b]['groups'] = disagreements_by_pair(m['df'])
    r = results[b]['res']
    print(f"===== {b}: n={r['n_valid']} | % đồng thuận={r['percent_agreement']} | κ 4 lớp={r['kappa_4class']}")
    print(f"  actionable–OTHER: % đồng thuận={r['actionable_other']['percent_agreement']} | κ={r['actionable_other']['kappa']}")
    print(f"  FEATURE–UX (cả hai chọn FEATURE/UX): n={r['feature_ux']['n_both_feature_or_ux']} | "
          f"% đồng thuận={r['feature_ux']['percent_agreement']} | κ={r['feature_ux']['kappa']}")
    print('  Ma trận nhầm lẫn (hàng = A, cột = B):')
    print(r['confusion'].to_string())

===== batch1: n=75 | % đồng thuận=88.0 | κ 4 lớp=0.671
  actionable–OTHER: % đồng thuận=90.7 | κ=0.728
  FEATURE–UX (cả hai chọn FEATURE/UX): n=2 | % đồng thuận=0.0 | κ=0.0
  Ma trận nhầm lẫn (hàng = A, cột = B):
B        BUG  FEATURE  UX  OTHER
A                               
BUG       11        0   0      0
FEATURE    0        0   2      1
UX         0        0   0      2
OTHER      2        2   0     55
===== batch2: n=75 | % đồng thuận=89.3 | κ 4 lớp=0.785
  actionable–OTHER: % đồng thuận=93.3 | κ=0.849
  FEATURE–UX (cả hai chọn FEATURE/UX): n=6 | % đồng thuận=100.0 | κ=1.0
  Ma trận nhầm lẫn (hàng = A, cột = B):
B        BUG  FEATURE  UX  OTHER
A                               
BUG       13        0   0      0
FEATURE    2        1   0      0
UX         1        0   5      0
OTHER      4        1   0     48


### Các ca hai người gán khác nhau

**Nhóm theo cặp nhãn**

Các ca lệch được nhóm theo cặp nhãn (ví dụ A chọn FEATURE, B chọn UX), kèm nội dung review và ghi chú của cả hai người, để mang vào
buổi họp sửa guideline.

In [5]:
for b, out in results.items():
    print(f'===== {b}')
    for (la, lb), rows in out['groups'].items():
        print(f'--- A={la} / B={lb}: {len(rows)} ca')
        for r in rows:
            print(f"  [{r['review_id'][:8]}] {r['text'][:200]}")
            if r['note_a'] or r['note_b']:
                print(f"      note A: {r['note_a'] or '-'} | note B: {r['note_b'] or '-'}")
if not results:
    print('Chưa có batch nào đủ hai tệp gán nhãn.')

===== batch1
--- A=OTHER / B=FEATURE: 2 ca
  [29248bd9] mong thêm nhiều giảm giá
      note A: Yêu cầu thêm giảm giá, thuộc khuyến mãi/chính sách | note B: Đề xuất thêm hoặc mở rộng giảm giá, là feature request
  [9e89d23d] bỏ thêm tủ đồ đi nhé, không đặt hàng dc
      note A: Yêu cầu thêm sản phẩm/tủ đồ, không phải yêu cầu sửa chức năng app | note B: Đề xuất bỏ hoặc thêm tủ đồ để đặt hàng được, là feature request
--- A=OTHER / B=BUG: 2 ca
  [42a8462c] mk vừa lập acc lưu đc viu chờ 80k xg e nó ghi là 3/8 hết hạn r mk để đấy chx đc 3 ngày viu chờ ko còn nó lại cho mk viu chờ 60k thôi bực ghê
      note A: Voucher/ưu đãi hết hạn hoặc thay đổi, thuộc chính sách khuyến mãi; chưa đủ căn cứ là lỗi app | note B: Voucher bị mất hoặc hết hạn sai, là lỗi voucher
  [8e0c8551] đang mua thì bị hết pin, bị thu dth, mong nhà phát hành sửa lỗi
      note A: Nêu tình huống thiết bị/không đủ thông tin về lỗi app cụ thể | note B: Lỗi trong quá trình mua như hết pin hoặc bị thu điện thoại, là malfunction


## Có chốt được guideline chưa

**Nội dung bảng tổng hợp**

κ tổng, κ ở hai chỗ dễ nhầm, số ca lệch theo từng nhóm và số ca được đánh dấu ambiguous.

**Nếu chưa đạt**

Nếu κ của batch 2 dưới 0.6, hoặc FEATURE và UX vẫn lệch nhau nhiều, guideline chưa được chốt: chính các ca lệch này được dùng để họp
sửa, rồi làm thêm một batch calibration nữa.

**Nếu đạt**

Bước chốt guideline (thêm các ví dụ thật từ calibration vào `annotation_guideline_v1.md`) được làm riêng sau khi xem lại kết quả.

In [6]:
for b, out in results.items():
    print(gate_checklist(out['res'], out['groups'], b, is_gate_batch=(b == GATE_BATCH)))
    print()
if not results:
    print('Chưa đủ dữ liệu để lập checklist.')

Checklist chốt guideline — batch1
  κ tổng (4 lớp)          : 0.671  (cổng ≥ 0.6)
  % đồng thuận            : 88.0%  (n=75)
  κ actionable–OTHER      : 0.728  | bất đồng: 7
  κ FEATURE–UX (tập con)  : 0.0  | n tập con: 2 | bất đồng FEATURE↔UX: 2
  Đánh dấu ambiguous      : A=1, B=1
  Số ca bất đồng theo cặp (A/B): OTHER/FEATURE=2, OTHER/BUG=2, UX/OTHER=2, FEATURE/UX=2, FEATURE/OTHER=1
  → Batch thảo luận: dùng các ca bất đồng ở trên để họp sửa guideline trước khi gán batch tiếp theo.

Checklist chốt guideline — batch2
  κ tổng (4 lớp)          : 0.785  (cổng ≥ 0.6)
  % đồng thuận            : 89.3%  (n=75)
  κ actionable–OTHER      : 0.849  | bất đồng: 5
  κ FEATURE–UX (tập con)  : 1.0  | n tập con: 6 | bất đồng FEATURE↔UX: 0
  Đánh dấu ambiguous      : A=1, B=1
  Số ca bất đồng theo cặp (A/B): OTHER/BUG=4, FEATURE/BUG=2, UX/BUG=1, OTHER/FEATURE=1
  → κ đạt cổng. Người nghiên cứu xem thêm 0 ca bất đồng FEATURE/UX trước khi kết luận; nếu không còn vấn đề hệ thống: guideline sẵn sàng chố

## Bộ bổ sung để kiểm tra ranh giới FEATURE–UX

**Vấn đề của batch ngẫu nhiên**

Batch 1 lấy ngẫu nhiên nên phần lớn là OTHER: chỉ khoảng 3–5 review được gán FEATURE hoặc UX, và κ ở ranh giới FEATURE–UX chỉ tính được
trên 2 ca, gần như không có ý nghĩa. Batch 2 nhiều khả năng cũng vậy, nên cổng κ không cho biết hai người có hiểu giống nhau ở chỗ này
hay không — trong khi đây lại đúng là chỗ vừa thêm quy tắc R3.

**Rút theo cụm từ**

Rút thêm khoảng 30 review có chứa các cụm từ hay gặp trong góp ý tính năng hoặc phàn nàn trải nghiệm ("nên thêm", "mong có", "bổ sung",
"khó dùng", "khó tìm", "không tìm thấy", "rối", "phiền", "cho tắt", "tùy chọn"), loại những review đã nằm trong batch 1 và batch 2.

**Rút đều theo nhóm keyword**

Lần rút đầu tiên lấy ngẫu nhiên trên toàn bộ review khớp keyword thì 27/30 review chứa chữ "phiền" (cụm này chiếm khoảng 80% số review
khớp, phần lớn là phàn nàn quảng cáo đã được R2 xử lý), nên cách rút được đổi sang rút đều 3 review cho mỗi nhóm keyword.

**Vai trò của bộ bổ sung**

Hai người gán thử độc lập bằng guideline v1.1 rồi cùng xem lại. Bộ này chỉ dùng để thảo luận ranh giới, **không** tính vào κ cổng; cổng
vẫn đo trên batch 2 lấy ngẫu nhiên (amendment v1.6).

**Loại khỏi Hold-out**

Người gán đã đọc các review này, nên chúng cũng được ghi manifest riêng để loại khỏi Hold-out sau này.

In [7]:
KW_SHEET = os.path.join(CAL_DIR, 'calibration_feature_ux_blank.csv')
KW_MANIFEST = os.path.join(input_dir('manifests'), f'calibration_feature_ux_manifest_{TAG}.json')
used_ids = [rid for info in manifest['batches'].values() for rid in info['review_ids']]
if os.path.exists(KW_SHEET) and os.path.exists(KW_MANIFEST):
    print('Bộ bổ sung FEATURE–UX đã tồn tại — giữ nguyên (không ghi đè).')
else:
    kw_rows, n_cand, hits = draw_keyword_set(records, n=30, exclude_ids=used_ids, seed=42)
    export_blank(kw_rows, KW_SHEET)
    write_keyword_manifest(kw_rows, FEATURE_UX_KEYWORDS, n_cand, SOURCE, KW_SHEET, KW_MANIFEST, seed=42)
    print(f'Số review khớp keyword (sau khi loại batch 1/2): {n_cand:,}')
    print('Số review rút theo từng nhóm keyword (3 mỗi nhóm):', hits)
with open(KW_MANIFEST, encoding='utf-8') as f:
    kw_manifest = json.load(f)
print(f"Bộ bổ sung: {len(kw_manifest['review_ids'])} review → {KW_SHEET}")
print('Trùng với batch 1/2:', len(set(kw_manifest['review_ids']) & set(used_ids)))

Bộ bổ sung FEATURE–UX đã tồn tại — giữ nguyên (không ghi đè).
Bộ bổ sung: 30 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\calibration\calibration_feature_ux_blank.csv
Trùng với batch 1/2: 0


### Đo trên bộ bổ sung (không phải κ cổng)

**Điều kiện chạy**

Phần này chạy khi đã có `calibration_feature_ux_A.csv` và `calibration_feature_ux_B.csv`.

**Vai trò**

Kết quả chỉ để xem hai người có tách FEATURE, UX và BUG giống nhau hay không; nó không được dùng để quyết định chốt guideline.

In [8]:
pa, pb = (os.path.join(CAL_DIR, f'calibration_feature_ux_{w}.csv') for w in ('A', 'B'))
if not (os.path.exists(pa) and os.path.exists(pb)):
    print('Bộ bổ sung FEATURE–UX: đang chờ calibration_feature_ux_A.csv và calibration_feature_ux_B.csv')
else:
    m = merge(pa, pb, expected_ids=kw_manifest['review_ids'])
    if m['problems']:
        print('Cần sửa tệp gán nhãn trước khi đo:')
        for pr in m['problems']:
            print('   -', pr)
    else:
        kw_res = agreement(m['df'])
        print('KHÔNG phải κ cổng — chỉ để thảo luận ranh giới FEATURE–UX.')
        print(f"n={kw_res['n_valid']} | % đồng thuận={kw_res['percent_agreement']} | κ 4 lớp={kw_res['kappa_4class']}")
        print(f"FEATURE–UX (cả hai chọn FEATURE/UX): n={kw_res['feature_ux']['n_both_feature_or_ux']} | "
              f"% đồng thuận={kw_res['feature_ux']['percent_agreement']} | κ={kw_res['feature_ux']['kappa']}")
        print('Ma trận nhầm lẫn FEATURE/UX/BUG (hàng = A, cột = B):')
        print(kw_res['confusion'].loc[['FEATURE', 'UX', 'BUG'], ['FEATURE', 'UX', 'BUG']].to_string())
        for (la, lb), rows in disagreements_by_pair(m['df']).items():
            print(f'--- A={la} / B={lb}: {len(rows)} ca')
            for r in rows:
                print(f"  [{r['review_id'][:8]}] {r['text'][:200]}")
                if r['note_a'] or r['note_b']:
                    print(f"      note A: {r['note_a'] or '-'} | note B: {r['note_b'] or '-'}")

KHÔNG phải κ cổng — chỉ để thảo luận ranh giới FEATURE–UX.
n=30 | % đồng thuận=86.7 | κ 4 lớp=0.814
FEATURE–UX (cả hai chọn FEATURE/UX): n=14 | % đồng thuận=92.9 | κ=0.851
Ma trận nhầm lẫn FEATURE/UX/BUG (hàng = A, cột = B):
B        FEATURE  UX  BUG
A                        
FEATURE        8   1    1
UX             0   5    1
BUG            0   1    2
--- A=FEATURE / B=BUG: 1 ca
  [7447906e] không cài face id mà thanh toán bắt dùng face id. cũng không cho tắt😡
      note A: Thanh toán bắt Face ID dù người dùng chưa cài và không có tùy chọn tắt; yêu cầu thêm lựa chọn xác thực là FEATURE. | note B: bat buoc dung face id du chua cai, khong cho tat, malfunction
--- A=FEATURE / B=UX: 1 ca
  [7825c8f5] bắt ép coi live và video khi mở app, lướt mua hàng tới đâu cũng hiện video gây khó chịu. không cho tắt và không có cách liên hệ hỗ trợ. cực kỳ phiền
      note A: Hành vi tự mở Live hoặc video kèm yêu cầu không cho tắt; theo R3, đề xuất thêm tùy chọn thay đổi là FEATURE. | note B: hanh vi da 

## Chốt guideline và chốt nhãn calibration

**Kết quả batch 2**

Batch 2 đạt cổng (κ = 0,785) và hai người không lệch nhau ở FEATURE/UX, nhưng 7/8 ca lệch lại dồn quanh BUG: một người hay gán
"malfunction" cho những thao tác app không hỗ trợ, hoặc cho kết quả do chính sách và giao hàng.

**Quy tắc R6 và guideline v1.2**

Hai người thống nhất thêm quy tắc R6 ("app làm sai điều lẽ ra làm được" là BUG, "app không làm được điều tôi muốn" là FEATURE, kết quả
do chính sách/chống gian lận/logistics là OTHER) và chốt luôn guideline ở bản v1.2 mà không gán thêm batch.

**Lý do không gán thêm batch**

κ đã vượt cổng khá xa và các ca lệch chỉ nằm ở một ranh giới; Hold-out sẽ được hai người gán toàn bộ nên R6 sẽ được kiểm lại bằng chính
độ đồng thuận ở đó.

**Rà lại nhãn 180 review calibration**

180 review calibration được đưa vào Train (Q9), nên nhãn của chúng phải khớp guideline đã đóng băng. Khi chọn ví dụ cho guideline, phát
hiện một số review hai người cùng gán giống nhau nhưng lại trái quy tắc mới — ví dụ quảng cáo bị gán BUG ở batch 1 (trước khi có R2),
hay đơn bị hủy vì "hoạt động bất thường" bị gán BUG trong khi R6 coi đây là quyết định của hệ thống chống gian lận.

**File chốt nhãn**

Toàn bộ 180 review được rà lại và một file được xuất gồm tất cả ca lệch cùng các ca đồng thuận nhưng trái v1.2
(`agreed_conflicts_v1.2`). Cột `R_rule` ghi quy tắc liên quan và nhãn đã chốt (nếu có) hoặc nhãn gợi ý; cột `agreed_label` để trống để
hai người điền nhãn cuối.

In [9]:
CAL_SETS = {s: (os.path.join(CAL_DIR, f'calibration_{s}_A.csv'), os.path.join(CAL_DIR, f'calibration_{s}_B.csv'))
            for s in ('batch1', 'batch2', 'feature_ux')}

# Ghi chú theo quy tắc cho từng ca lệch (mã = 8 ký tự đầu review_id). "đã chốt" = đã thống nhất, ghi trong edge_cases_log.md
RULES = {
    # batch 1 (đã chốt, guideline v1.1)
    '29248bd9': 'R1 → OTHER (đã chốt)', '42a8462c': 'R1 → OTHER (đã chốt)',
    '42acd7bc': 'R2 → UX (đã chốt)', '79bd9683': 'R2 → UX (đã chốt)',
    'dad71cb2': 'R3 → FEATURE (đã chốt)', 'fb0c4bbc': 'R3 + quy tắc ambiguous → FEATURE, ambiguous=1 (đã chốt)',
    '93d24e85': 'R4 → FEATURE (đã chốt)', '8e0c8551': 'R5 → OTHER (đã chốt)', '9e89d23d': 'R5 → OTHER (đã chốt)',
    # batch 2 (đã chốt, guideline v1.2)
    '92402354': 'R6 → FEATURE (đã chốt)', 'e63ed42d': 'R6 → FEATURE (đã chốt)', 'aa4bf2af': 'R6 → OTHER (đã chốt)',
    '99aa591f': 'R6 → OTHER (đã chốt)', '75be2c08': 'R6 → OTHER (đã chốt)', '481654e7': 'R5 → OTHER (đã chốt)',
    # bộ bổ sung (đã chốt, guideline v1.2)
    '7447906e': 'R6 + R3 bổ sung → FEATURE (đã chốt)', '7825c8f5': 'R3 bổ sung → FEATURE (đã chốt)',
    # CHƯA chốt — chỉ gợi ý quy tắc để hai người thảo luận
    '63848f1a': 'CHƯA CHỐT — quy tắc ambiguous: nhiều ý khác lớp; hai người khác nhau về lớp của ý đầu '
                '(tin nhắn lừa đảo vẫn đến: UX theo R2 hay BUG?)',
    'cf7471aa': 'CHƯA CHỐT — R4 (yêu cầu đổi phương thức hoàn tiền → FEATURE) hay R6 (chính sách hoàn tiền → OTHER)?',
    '9d8a3972': 'CHƯA CHỐT — quy tắc ambiguous: không tìm thấy mục sửa địa chỉ (UX) + "mã giảm lỗi tùm lum" (BUG hay R5 mơ hồ?)',
    'c6662107': 'CHƯA CHỐT — "đừng làm rối địa chỉ … trả lại đơn": R5 mơ hồ (OTHER), BUG (xử lý địa chỉ sai) hay UX?',
}

# Ca hai người ĐỒNG THUẬN nhưng nhãn trái guideline v1.2 — nhãn bên phải chỉ là GỢI Ý
AGREED_CONFLICTS = {
    # R6 (làm rõ 12/10): hệ thống chống gian lận / chính sách → OTHER
    'f8fce12e': 'R6 — đơn tự hủy khi áp voucher (chống gian lận) → gợi ý OTHER',
    '93c18679': 'R6 — tự động hủy đơn + không cho trả hàng (chính sách) → gợi ý OTHER',
    'b6d4b63e': 'R6 — "voucher ảo" rồi khóa sđt (chống gian lận) → gợi ý OTHER',
    'da1855df': 'R6 — báo lỗi rồi khóa tài khoản (chính sách) → gợi ý OTHER',
    '62690671': 'R6 — không nhận xu vì "hoạt động bất thường" → gợi ý OTHER',
    'c42ebf79': 'R6 — "giao dịch bất thường", không đặt được → gợi ý OTHER',
    'd076ab22': 'R6 — tự hủy đơn vì "bất thường" → gợi ý OTHER',
    '296f3681': 'R6 — đơn toàn tự bị hủy (chống gian lận) → gợi ý OTHER',
    '0f8c12ea': 'R6 — đặt gì cũng hủy "kêu bất thường" → gợi ý OTHER',
    '29f95921': 'R6 — có voucher xong đặt thì hủy → gợi ý OTHER',
    '682e8322': 'R6 — "ko cho trả hàng" (chính sách); batch 1 có 87de4734 "ko cho hoàn hàng" đã đồng thuận OTHER → gợi ý OTHER',
    # R2: quảng cáo / tự chuyển sang app → UX
    '8466ce83': 'R2 — đang xem phim cứ tự nhảy sang Shopee (quảng cáo chuyển app) → gợi ý UX',
    'd093c4f1': 'R2 — quảng cáo hiện liên tục, tự chuyển sang app → gợi ý UX',
    # ranh giới — cần hai người cân nhắc
    'cce5f87f': 'RANH GIỚI — "k mua được": R5 (mơ hồ → OTHER) hay BUG (không mua được, như "không đăng nhập được" của Q16)?',
    '67b0d008': 'RANH GIỚI — "dính m02": mã lỗi kỹ thuật (BUG) hay mã hạn chế tài khoản (R6 → OTHER)?',
    '2161fc14': 'RANH GIỚI — SPayLater hoàn vào số dư Shopee không rút được: chính sách (R6 → OTHER) hay BUG?',
}

FINALIZE = os.path.join(CAL_DIR, 'disagreements_to_finalize.csv')
fin_rows = build_finalize_rows(CAL_SETS, RULES, AGREED_CONFLICTS)
if os.path.exists(FINALIZE):
    print('File chốt nhãn đã tồn tại — giữ nguyên (có thể đã điền agreed_label).')
else:
    write_finalize(fin_rows, FINALIZE)
    print('Đã xuất:', FINALIZE)
fin = pd.DataFrame(fin_rows)
print(fin['source'].value_counts().to_string())
print('Chưa chốt (cần thảo luận):', int(fin['R_rule'].str.startswith('CHƯA CHỐT').sum()),
      '| ranh giới:', int(fin['R_rule'].str.startswith('RANH GIỚI').sum()))

File chốt nhãn đã tồn tại — giữ nguyên (có thể đã điền agreed_label).
source
agreed_conflicts_v1.2      16
batch1_disagreement         9
batch2_disagreement         8
feature_ux_disagreement     4
Chưa chốt (cần thảo luận): 4 | ranh giới: 3


## Ghi nhãn cuối của 180 review calibration vào gold

**Nguồn nhãn cuối**

Sau khi hai người điền xong `agreed_label` và `agreed_ambiguous` cho 37 ca trong file chốt nhãn, nhãn cuối được dựng cho toàn bộ 180
review calibration (batch 1, batch 2 và bộ bổ sung). Review nào có trong file chốt nhãn thì lấy nhãn đã thống nhất (kể cả các ca trước
đó hai người gán giống nhau nhưng trái guideline v1.2); các review còn lại là những ca hai người gán giống nhau và khớp v1.2, nên lấy
luôn nhãn đồng thuận.

**Cờ ambiguous**

Với các ca đồng thuận, `ambiguous_flag = 1` chỉ được ghi khi cả hai người cùng đánh dấu.

**File dùng để train**

`gold/adjudicated/calibration_gold_v1.2.csv` chỉ có văn bản và nhãn, không có số sao hay ngày (Q17, và đề cương dùng văn bản làm đầu
vào duy nhất).

**File truy vết**

`gold/provenance.csv` lưu đầy đủ truy vết theo Phụ lục B của đề cương: nhãn của từng người, nhãn cuối, cách ra nhãn (đồng thuận hay hội
ý), phiên bản guideline, cùng số sao và ngày review làm metadata cho phân tích thời gian sau này.

**Loại khỏi Hold-out**

Cả 180 review được ghi vào danh sách loại khỏi Hold-out (Q9).

In [10]:
from datetime import date
from src.annotation.gold import (load_finalized, build_calibration_gold, write_provenance, write_train_gold,
                                 write_holdout_exclusion, PROVENANCE_COLUMNS)
GOLD_DIR = input_dir('gold')
finalized = load_finalized(FINALIZE)
print(f'File chốt nhãn: {len(finalized)} ca, nhãn hợp lệ, không ô trống')
print('  agreed_label:', pd.Series([v[0] for v in finalized.values()]).value_counts().to_dict(),
      '| agreed_ambiguous = 1:', sum(v[1] for v in finalized.values()))

interim_by_id = {r['review_id']: r for r in records}
gold_rows, gstats = build_calibration_gold(CAL_SETS, finalized, interim_by_id, guideline_version='v1.2',
                                           label_date=date.today().isoformat())
g = pd.DataFrame(gold_rows)
print(f"\nNhãn cuối: {len(g)} review | từ file chốt nhãn: {gstats['from_finalize']} | đồng thuận khớp v1.2: {gstats['consensus']}")
print('Theo bộ:', g['annotation_set'].value_counts().to_dict())
print('Phân bố nhãn cuối:', g['gold_label'].value_counts().reindex(['BUG', 'FEATURE', 'UX', 'OTHER']).to_dict(),
      '| ambiguous_flag = 1:', int(g['ambiguous_flag'].sum()))
print(pd.crosstab(g['annotation_set'], g['gold_label']).reindex(columns=['BUG', 'FEATURE', 'UX', 'OTHER'], fill_value=0).to_string())
if gstats['ambiguous_one_sided']:
    print('Ca đồng thuận nhãn nhưng chỉ MỘT người đánh ambiguous (ghi ambiguous_flag = 0):',
          [x[:8] for x in gstats['ambiguous_one_sided']])

File chốt nhãn: 37 ca, nhãn hợp lệ, không ô trống
  agreed_label: {'OTHER': 23, 'FEATURE': 8, 'UX': 6} | agreed_ambiguous = 1: 3

Nhãn cuối: 180 review | từ file chốt nhãn: 37 | đồng thuận khớp v1.2: 143
Theo bộ: {'batch1': 75, 'batch2': 75, 'feature_ux': 30}
Phân bố nhãn cuối: {'BUG': 10, 'FEATURE': 17, 'UX': 16, 'OTHER': 137} | ambiguous_flag = 1: 6
gold_label      BUG  FEATURE  UX  OTHER
annotation_set                         
batch1            2        3   4     66
batch2            6        4   6     59
feature_ux        2       10   6     12


In [11]:
PROVENANCE = os.path.join(GOLD_DIR, 'provenance.csv')
TRAIN_GOLD = os.path.join(GOLD_DIR, 'adjudicated', 'calibration_gold_v1.2.csv')
EXCLUSION = os.path.join(input_dir('manifests'), f'holdout_exclusion_{TAG}.json')
if os.path.exists(TRAIN_GOLD):
    print('Gold calibration đã được ghi trước đó — giữ nguyên.')
else:
    write_train_gold(gold_rows, TRAIN_GOLD)
    write_provenance(gold_rows, PROVENANCE)
    write_holdout_exclusion(gold_rows, EXCLUSION)
prov = pd.read_csv(PROVENANCE, encoding='utf-8-sig')
excl = json.load(open(EXCLUSION, encoding='utf-8'))
print('provenance.csv:', len(prov), 'dòng | đủ cột theo Phụ lục B + truy vết:', list(prov.columns) == PROVENANCE_COLUMNS,
      '| có ambiguous_flag:', 'ambiguous_flag' in prov.columns)
print('Cột:', list(prov.columns))
print('Train gold:', TRAIN_GOLD, '| cột:', list(pd.read_csv(TRAIN_GOLD, encoding='utf-8-sig').columns))
print(f"Loại khỏi Hold-out: {excl['n']} review_id — {excl['by_set']}")

Gold calibration đã được ghi trước đó — giữ nguyên.
provenance.csv: 580 dòng | đủ cột theo Phụ lục B + truy vết: True | có ambiguous_flag: True
Cột: ['review_id', 'app', 'content', 'raw_text_hash', 'rating', 'review_date', 'app_version', 'sampling_source', 'annotation_set', 'split', 'label_a', 'label_b', 'agreement_flag', 'gold_label', 'ambiguous_flag', 'guideline_version', 'annotated_with_guideline', 'labeled_by', 'resolution', 'label_date']
Train gold: C:\Workspace\kltn-vn-app-review-requirements\01_data\gold\adjudicated\calibration_gold_v1.2.csv | cột: ['review_id', 'app', 'text', 'gold_label', 'ambiguous_flag', 'sampling_source', 'split', 'guideline_version']
Loại khỏi Hold-out: 180 review_id — {'batch1': 75, 'batch2': 75, 'feature_ux': 30}


## Đợt gán diện rộng (GĐ4b): kiểm tra file và đo độ đồng thuận

**Các tập gán đôi**

Sau khi chia tập (notebook 03, amendment v1.7), hai người đã gán độc lập Hold-out phần rút thêm (600), Validation (350), focused
FEATURE/UX (150) và Lazada (500).

**Train và audit**

Người gán chính gán một mình 1.020 review Train mới; người gán thứ hai gán lại 153 review trong số đó (15%) để kiểm tra chất lượng nhãn
Train.

**Kiểm tra file trước khi đo**

Mỗi file được kiểm: đúng tập review theo manifest, không ô trống, nhãn hợp lệ, cột ambiguous chỉ có 0 hoặc 1. Sau đó độ đồng thuận được
đo cho từng tập giống như calibration.

**κ audit của Train**

Với Train, nhãn của người gán chính được ghép với nhãn audit trên 153 review để có κ audit. Các ca lệch ở đây chỉ dùng để đánh giá chất
lượng; nhãn Train không bị sửa hàng loạt theo audit vì Train là tập một người gán.

**Lazada được xem riêng**

Lazada có phân bố khác Shopee (nhiều review 1★ hơn) nên κ của Lazada được xem riêng, không coi là lỗi nếu khác.

In [12]:
from src.annotation.wide import check_pair, check_single, audit_frame, summary_row, warnings_for
ANN_DIR = os.path.join(input_dir('gold'), 'annotations')
split_manifest = json.load(open(os.path.join(input_dir('manifests'), f'splits_manifest_{TAG}.json'), encoding='utf-8'))
S = split_manifest['sets']
DOUBLE = {  # tên tập → (tiền tố file, review_id cần gán)
    'holdout_extra_600': ('holdout_extra_600', S['holdout']['to_label']),
    'validation': ('validation', S['validation']['to_label']),
    'focused_feature_ux': ('focused_feature_ux', S['focused']['to_label']),
    'lazada': ('lazada', S['lazada_external_test']['to_label']),
}
checks = {}
for name, (prefix, ids) in DOUBLE.items():
    pa, pb = (os.path.join(ANN_DIR, f'{prefix}_{w}.csv') for w in ('A', 'B'))
    checks[name] = check_pair(pa, pb, ids)
    status = 'OK' if not checks[name]['problems'] else 'CÓ LỖI'
    print(f"{name:20s} {len(checks[name]['df'])} review | {status}")
    for pr in checks[name]['problems']:
        print('    -', pr)
TRAIN_SHEET = os.path.join(ANN_DIR, 'train_extra.csv')
AUDIT_SHEET_B = os.path.join(ANN_DIR, 'train_audit_B.csv')
tr_check = check_single(TRAIN_SHEET, S['train']['to_label'])
print(f"{'train_extra':20s} {tr_check['n']} review | {'OK' if not tr_check['problems'] else 'CÓ LỖI'}")
for pr in tr_check['problems']:
    print('    -', pr)
audit = audit_frame(TRAIN_SHEET, AUDIT_SHEET_B)
print(f"{'train_audit':20s} {audit['n_audit_in_train']}/{audit['n_audit']} review audit nằm trong Train"
      f" | khớp danh sách audit trong manifest: {set(audit['df']['review_id']) == set(S['train']['audit_ids'])}"
      f" | {'OK' if not audit['problems'] else 'CÓ LỖI'}")
for pr in audit['problems']:
    print('    -', pr)

holdout_extra_600    600 review | OK
validation           350 review | OK
focused_feature_ux   150 review | OK


lazada               500 review | OK
train_extra          1020 review | OK
train_audit          153/153 review audit nằm trong Train | khớp danh sách audit trong manifest: True | OK


In [13]:
results, summary, warns = {}, [], []
measured = {**{k: v['df'] for k, v in checks.items() if not v['problems']},
            **({'train_audit': audit['df']} if not audit['problems'] else {})}
for name, df in measured.items():
    res = agreement(df)
    grp = disagreements_by_pair(df)
    results[name] = (res, grp)
    summary.append(summary_row(name, res, grp))
    warns += warnings_for(name, res, is_external=(name == 'lazada'))
print('Bảng tổng (train_audit: A = người gán chính, B = người audit):')
print(pd.DataFrame(summary).to_string(index=False))
print('\nCảnh báo:' if warns else '\nKhông có cảnh báo (κ tổng ≥ 0,6 và κ FEATURE–UX không tụt quá 0,15 so với pilot).')
for w in warns:
    print('  -', w)

Bảng tổng (train_audit: A = người gán chính, B = người audit):
               tập   n  % đồng thuận  κ tổng  κ act–OTHER  κ FEAT–UX  n FEAT–UX  số ca lệch
 holdout_extra_600 600          91.8   0.752        0.752      0.946         47          49
        validation 350          94.6   0.816        0.846      0.865         25          19
focused_feature_ux 150          74.0   0.619        0.545      0.902         75          39
            lazada 500          92.2   0.745        0.777      0.755         34          39
       train_audit 153          86.3   0.741        0.743      0.893         40          21

Không có cảnh báo (κ tổng ≥ 0,6 và κ FEATURE–UX không tụt quá 0,15 so với pilot).


In [14]:
for name, (res, grp) in results.items():
    print(f'===== {name}')
    print(res['confusion'].to_string())
    for (la, lb), rows in grp.items():
        print(f'--- A={la} / B={lb}: {len(rows)} ca')
        for r in rows:
            print(f"  [{r['review_id'][:8]}] {r['text'][:160]}")
            if r['note_a'] or r['note_b']:
                print(f"      note A: {r['note_a'] or '-'} | note B: {r['note_b'] or '-'}")
    print()

===== holdout_extra_600
B        BUG  FEATURE  UX  OTHER
A                               
BUG       42        1   1      6
FEATURE    0       12   1      2
UX         0        0  34      1
OTHER     29        2   6    463
--- A=OTHER / B=BUG: 29 ca
  [14f2f2b1] như l k cho đặt hàng
      note A: Ngoài lỗi hoặc chức năng app, không cụ thể, hoặc do chính sách hoặc dịch vụ | note B: -
  [171efc03] app chơi ngang như cua
      note A: Ngoài lỗi hoặc chức năng app, không cụ thể, hoặc do chính sách hoặc dịch vụ | note B: -
  [2942b174] số điện thoại chính chủ tạo tài khoản thì yêu cầu xác minh . xác minh thì có tài khoản người khác tạo rồi . khôi phục thì yêu cấu xác minh thêm email . email củ
      note A: Ngoài lỗi hoặc chức năng app, không cụ thể, hoặc do chính sách hoặc dịch vụ | note B: -
  [32ab5acf] gấp muốn thấy me 1 cái ko cho. hết xu luôn. mấy bạn coi chừng nó nghe cay. khó ua
      note A: Ngoài lỗi hoặc chức năng app, không cụ thể, hoặc do chính sách hoặc dịch vụ | note B: -
  [3

### Xuất file chốt ca lệch

**File cho từng tập gán đôi**

Mỗi tập gán đôi có ca lệch được xuất thành một file `<tập>_disagreements_to_finalize.csv` để hai người điền `agreed_label` và
`agreed_ambiguous` theo guideline v1.2.

**Tập khóa**

Hold-out, focused và Lazada là tập khóa nên khi chốt chỉ áp guideline đã đóng băng.

**Train audit**

Ca lệch của Train audit không xuất file, vì chúng chỉ dùng để đánh giá chất lượng nhãn Train.

In [15]:
exported = []
for name, (prefix, _) in DOUBLE.items():
    if name not in results or not results[name][1]:
        continue
    out = os.path.join(ANN_DIR, f'{prefix}_disagreements_to_finalize.csv')
    if os.path.exists(out):
        print('Đã có, giữ nguyên:', os.path.basename(out))
    else:
        pa, pb = (os.path.join(ANN_DIR, f'{prefix}_{w}.csv') for w in ('A', 'B'))
        write_finalize(build_finalize_rows({name: (pa, pb)}, {}, {}), out)
    exported.append((os.path.basename(out), sum(len(v) for v in results[name][1].values())))
for f, n in exported:
    print(f'{f:55s} {n} ca')

Đã có, giữ nguyên: holdout_extra_600_disagreements_to_finalize.csv
Đã có, giữ nguyên: validation_disagreements_to_finalize.csv
Đã có, giữ nguyên: focused_feature_ux_disagreements_to_finalize.csv
Đã có, giữ nguyên: lazada_disagreements_to_finalize.csv
holdout_extra_600_disagreements_to_finalize.csv         49 ca
validation_disagreements_to_finalize.csv                19 ca
focused_feature_ux_disagreements_to_finalize.csv        39 ca
lazada_disagreements_to_finalize.csv                    39 ca


## Ghi nhãn cuối của đợt gán diện rộng

**Ca lệch đã chốt**

Hai người đã chốt 146 ca lệch của bốn tập gán đôi theo guideline v1.2 (cùng hai điểm diễn giải ghi trong edge_cases_log ngày 22/10).

**Nhãn cuối của tập gán đôi**

Nhãn cuối là nhãn đồng thuận khi A = B và nhãn đã chốt khi A ≠ B; `ambiguous_flag = 1` khi cả hai cùng đánh dấu hoặc khi đã chốt là 1.

**Nhãn cuối của Train**

Nhãn cuối là nhãn của người gán chính; nhãn audit được giữ trong `label_b` cho 15% review được kiểm tra, để truy vết, chứ không thay
nhãn Train.

**Ghi file**

Tất cả được ghi vào `gold/provenance.csv` theo đúng `split`. Các file dùng để mô hình hóa được ghi vào `01_data/processed/`, chỉ gồm
`review_id`, văn bản (`text_clean` — bản chuẩn hóa tối thiểu từ Giai đoạn 2), nhãn, `ambiguous_flag` và `split`; số sao, ngày và phiên
bản app nằm ở một file metadata riêng để dùng cho phân tích thời gian.

In [16]:
from src.annotation.gold import (build_double_label_gold, build_single_label_gold, write_provenance,
                                 write_processed, write_metadata)
PROCESSED = input_dir('processed')
interim_all = {**{r['review_id']: r for r in records},
               **{r['review_id']: r for r in load_jsonl(os.path.join(input_dir('interim'), f'lazada_clean_{TAG}.jsonl'))}}
today = date.today().isoformat()
NEW_SETS = {  # tên → (tiền tố file, split, sampling_source)
    'holdout_extra_600': ('holdout_extra_600', 'internal_holdout', 'random'),
    'validation': ('validation', 'validation', 'random'),
    'focused_feature_ux': ('focused_feature_ux', 'focused', 'keyword_enriched'),
    'lazada': ('lazada', 'lazada_external', 'random'),
}
prov = pd.read_csv(PROVENANCE, encoding='utf-8-sig', dtype=str)
have = set(prov['review_id'])
new_rows = []
for name, (prefix, split, src) in NEW_SETS.items():
    pa, pb = (os.path.join(ANN_DIR, f'{prefix}_{w}.csv') for w in ('A', 'B'))
    fin = os.path.join(ANN_DIR, f'{prefix}_disagreements_to_finalize.csv')
    rows, st = build_double_label_gold(name, pa, pb, fin, interim_all, sampling_source=src, split=split,
                                       annotated_with='v1.2', guideline_version='v1.2', label_date=today)
    print(f"{name:20s} {len(rows)} review | đồng thuận {st['consensus']} | đã chốt {st['from_finalize']}"
          f" | chỉ một người đánh ambiguous: {len(st['ambiguous_one_sided'])}")
    new_rows += [r for r in rows if r['review_id'] not in have]
tr_src = S['train']['sampling_source']
tr_rows, tst = build_single_label_gold('train_extra', TRAIN_SHEET, AUDIT_SHEET_B, interim_all, tr_src, split='train',
                                       guideline_version='v1.2', label_date=today)
print(f"{'train_extra':20s} {tst['n']} review | audit {tst['audited']} (khớp nhãn {tst['audit_agree']})")
new_rows += [r for r in tr_rows if r['review_id'] not in have]
if new_rows:
    write_provenance(new_rows, PROVENANCE)
    print('Đã bổ sung vào provenance:', len(new_rows), 'dòng')
else:
    print('Provenance đã đủ — không ghi thêm.')
prov = pd.read_csv(PROVENANCE, encoding='utf-8-sig', dtype=str)
print('provenance.csv:', len(prov), 'dòng')
print(prov.groupby(['split', 'annotation_set']).size().to_string())

holdout_extra_600    600 review | đồng thuận 551 | đã chốt 49 | chỉ một người đánh ambiguous: 1


validation           350 review | đồng thuận 331 | đã chốt 19 | chỉ một người đánh ambiguous: 1
focused_feature_ux   150 review | đồng thuận 111 | đã chốt 39 | chỉ một người đánh ambiguous: 7
lazada               500 review | đồng thuận 461 | đã chốt 39 | chỉ một người đánh ambiguous: 3
train_extra          1020 review | audit 153 (khớp nhãn 132)
Đã bổ sung vào provenance: 2620 dòng


provenance.csv:

 3200 dòng
split             annotation_set    
focused           focused_feature_ux     150
internal_holdout  holdout_extra_600      600
                  pilot                  400
lazada_external   lazada                 500
train             batch1                  75
                  batch2                  75
                  feature_ux              30
                  train_extra           1020
validation        validation             350


In [17]:
# File cho mô hình hóa + metadata riêng (01_data/processed/, đã gitignore)
SPLIT_FILES = {'train': 'shopee_train.csv', 'validation': 'shopee_validation.csv', 'internal_holdout': 'shopee_holdout.csv',
               'focused': 'shopee_focused_feature_ux.csv', 'lazada_external': 'lazada_external_test.csv'}
prov_rows = prov.to_dict('records')
for split, fname in SPLIT_FILES.items():
    write_processed([r for r in prov_rows if r['split'] == split], interim_all, os.path.join(PROCESSED, fname))
write_metadata(prov_rows, os.path.join(PROCESSED, f'labeled_metadata_{TAG}.csv'))
print('Đã ghi:', ', '.join(list(SPLIT_FILES.values()) + [f'labeled_metadata_{TAG}.csv']))

Đã ghi: shopee_train.csv, shopee_validation.csv, shopee_holdout.csv, shopee_focused_feature_ux.csv, lazada_external_test.csv, labeled_metadata_T0_2026-10-06.csv


## Nghiệm thu bộ dữ liệu có nhãn

In [18]:
order = ['BUG', 'FEATURE', 'UX', 'OTHER']
tabs = {}
for split, fname in SPLIT_FILES.items():
    d = pd.read_csv(os.path.join(PROCESSED, fname), encoding='utf-8-sig', dtype=str)
    tabs[split] = d
summary = pd.DataFrame({s: d['gold_label'].value_counts().reindex(order, fill_value=0) for s, d in tabs.items()}).T
summary['cỡ'] = [len(d) for d in tabs.values()]
summary['actionable %'] = ((summary[['BUG', 'FEATURE', 'UX']].sum(axis=1) / summary['cỡ']) * 100).round(1)
summary['ambiguous'] = [int((d['ambiguous_flag'] == '1').sum()) for d in tabs.values()]
print(summary.to_string())
ids = {s: set(d['review_id']) for s, d in tabs.items()}
from itertools import combinations
ov = {f'{a} ∩ {b}': len(ids[a] & ids[b]) for a, b in combinations(ids, 2)}
print('\nGiao nhau review_id giữa các split:', ov)
assert not any(ov.values()), 'LEAKAGE giữa các split'
assert all(len(ids[s]) == len(tabs[s]) for s in tabs), 'review_id trùng trong một split'
print('→ Không có review_id trùng giữa các split, không trùng trong từng split.')
print('Hold-out = 1.000:', len(tabs['internal_holdout']) == 1000, '| Train = 1.200:', len(tabs['train']) == 1200)

gold_label        BUG  FEATURE   UX  OTHER    cỡ  actionable %  ambiguous
train              75      144  132    849  1200          29.2         17
validation         27        7   22    294   350          16.0          0
internal_holdout   89       24   70    817  1000          18.3          1
focused            10       56   25     59   150          60.7          6
lazada_external    39       17   25    419   500          16.2          1

Giao nhau review_id giữa các split: {'train ∩ validation': 0, 'train ∩ internal_holdout': 0, 'train ∩ focused': 0, 'train ∩ lazada_external': 0, 'validation ∩ internal_holdout': 0, 'validation ∩ focused': 0, 'validation ∩ lazada_external': 0, 'internal_holdout ∩ focused': 0, 'internal_holdout ∩ lazada_external': 0, 'focused ∩ lazada_external': 0}
→ Không có review_id trùng giữa các split, không trùng trong từng split.
Hold-out = 1.000: True | Train = 1.200: True


In [19]:
# Validation có đủ lớp hiếm để tune không?
va = tabs['validation']['gold_label'].value_counts().reindex(order, fill_value=0)
print('Validation:', va.to_dict(), '| actionable:', int(va[['BUG', 'FEATURE', 'UX']].sum()))
MIN_PER_CLASS = 20   # ngưỡng tham khảo để per-class F1 trên Validation còn có nghĩa khi chọn ngưỡng/hyperparameter
low = {c: int(va[c]) for c in ['BUG', 'FEATURE', 'UX'] if va[c] < MIN_PER_CLASS}
if low:
    print(f'CẢNH BÁO: lớp dưới {MIN_PER_CLASS} ca trên Validation: {low}. Per-class F1 của các lớp này sẽ rất dao động khi tune.')
    print('Đề xuất (chưa làm): bổ sung Validation theo Q8 (stratified) — rút thêm từ pool còn lại, ưu tiên review khớp cụm từ')
    print('FEATURE, hai người gán, cho tới khi mỗi lớp actionable có khoảng 20–30 ca; ghi amendment.')
else:
    print('Mọi lớp actionable trên Validation đều ≥', MIN_PER_CLASS, 'ca.')

Validation: {'BUG': 27, 'FEATURE': 7, 'UX': 22, 'OTHER': 294} | actionable: 56
CẢNH BÁO: lớp dưới 20 ca trên Validation: {'FEATURE': 7}. Per-class F1 của các lớp này sẽ rất dao động khi tune.
Đề xuất (chưa làm): bổ sung Validation theo Q8 (stratified) — rút thêm từ pool còn lại, ưu tiên review khớp cụm từ
FEATURE, hai người gán, cho tới khi mỗi lớp actionable có khoảng 20–30 ca; ghi amendment.
